# Treinando modelo

Abrindo base de dados

In [ ]:
import pandas
from keras.src.layers.activations import activation
from keras.src.layers.pooling.global_average_pooling1d import GlobalAveragePooling1D
from keras.src.layers.pooling.global_average_pooling2d import GlobalAveragePooling2D

data_frame = pandas.read_csv('Dataset/processed/data.csv')
data_frame

Definindo função para o pré-processamento de imagens

In [ ]:
import numpy
from PIL import Image
import tensorflow

def preprocesar_imagem(path: str) -> None:
    # Carregando imagem e redimensionando para o tamanho esperado pelo ResNet50
    imagem = numpy.array(Image.open(path).convert('RGB').resize((224, 224)))

    # Pré-Processando a imagem de acordo com o ResNet50
    imagem = tensorflow.keras.applications.resnet50.preprocess_input(imagem)

    return imagem

Separando os dados de X e Y

In [ ]:
X = data_frame['file_path']
Y = data_frame['class']

Separando os dados de teste e treino

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=1024)

Criando modelo

In [ ]:
from keras.src.applications.resnet import ResNet50

base_model = ResNet50(weights='imagenet', include_top=False)

Adicionando camadas personalizadas

In [ ]:
from keras.src.layers import Dense
from  keras.src.layers.pooling.global_average_pooling2d import GlobalAveragePooling2D

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(len(data_frame['class'].unique()), activation='softmax')(x)

Definindo modelo completo

In [ ]:
from keras import Model

model = Model(inputs=base_model.input, outputs=predictions)

Congelando as camadas do modelo base

In [ ]:
for layer in base_model.layers:
    layer.treinable = False

Compilando o modelo

In [ ]:
from keras.src.optimizers import Adam

model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

Treinando o modelo ResNet50

In [ ]:
for caminho_imagem in x_train:
    imagem = preprocesar_imagem(caminho_imagem)
    model.fit(imagem)